# ViTASA Enhanced — Pair Classification (Colab Free)

Run on Google Colab Free GPU (T4) with safe time limits.

**Formulation:** target-aspect pair classification (correct)
**Backbone:** ViSoBERT (pre-trained on social media)
**Metric:** macro F1 on 3 sentiment classes (loại 'none')

⏱️ **Thời gian an toàn cho Colab Free:**
- Baseline C1 đơn domain: ~15 phút ✅ SAFE
- Full ablation (4 configs × 3 domains × 10 epochs): ~2 tiếng ✅ SAFE
- Full ablation × 20 epochs: ~4 tiếng (có rủi ro timeout)

**Strategy:**
1. Chạy C1 (baseline) trước để xác nhận formulation đúng
2. Nếu OK → chạy C2-C4 từng domain 1 lần (tránh timeout)
3. Nếu timeout → giảm epochs hoặc chạy lại domain bị interrupt


In [ ]:
# 1. Clone dataset từ ViTASA repo gốc
!git clone https://github.com/kh4nh12/ViTASA.git ViTASA_repo 2>&1 | grep -E '(Cloning|clone|done)'
!echo "Dataset files:"
!ls -lh ViTASA_repo/*.jsonl

In [ ]:
# 2. Install dependencies
!pip install -q torch transformers scikit-learn seqeval underthesea
import torch
print(f"✅ PyTorch {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# 3. Setup folder structure + copy dataset
import os
import shutil

os.makedirs('VITASA_Enhanced/baseline/data', exist_ok=True)
os.makedirs('VITASA_Enhanced/experiments/results_pair', exist_ok=True)

for domain in ['mobile', 'restaurant', 'hotel']:
    domain_dir = f'VITASA_Enhanced/baseline/data/{domain}'
    os.makedirs(domain_dir, exist_ok=True)
    src = f'ViTASA_repo/{domain}.jsonl'
    dst = f'{domain_dir}/{domain}.jsonl'
    if os.path.exists(src):
        shutil.copy(src, dst)
        lines = sum(1 for _ in open(dst))
        print(f"✅ {domain}: {lines} samples")

In [ ]:
# 4. Lấy code mới nhất từ GitHub (public repo — không cần Drive, không cần token)
# Mỗi lần bạn sửa code + git push, chỉ cần chạy lại CELL NÀY để lấy bản mới nhất,
# không phải kéo-thả file thủ công vào Drive nữa.
import os

REPO_URL = "https://github.com/Hunganh1305/VITASA_Enhanced.git"

if os.path.isdir("VITASA_Enhanced/.git"):
    !cd VITASA_Enhanced && git pull
else:
    !rm -rf VITASA_Enhanced_code_tmp
    !git clone {REPO_URL} VITASA_Enhanced_code_tmp
    # Merge vào folder VITASA_Enhanced (đã có sẵn baseline/data từ cell 3)
    !rsync -a VITASA_Enhanced_code_tmp/ VITASA_Enhanced/ --exclude 'baseline/data'
    !rm -rf VITASA_Enhanced_code_tmp

!echo "Files in VITASA_Enhanced:" && ls VITASA_Enhanced/ | grep -E '(train_pair|train_vitasd|text_norm|imbalanced)'

In [ ]:
# 5. Smoke test: 1 epoch trên mobile
%cd VITASA_Enhanced

!python3 train_pair.py --domain mobile --loss ce --epochs 1 --subsample 0.1 --batch-size 64 --fp16 2>&1 | tail -30

print("\n✅ Smoke test passed!")

In [ ]:
# 6. STRATEGY 1: Run BASELINE ONLY (C1) — safest, ~45 min for 3 domains
# ✅ Resume-safe: bỏ qua domain đã có results.json (an toàn khi chạy lại sau khi
#    bị ngắt/hết quota). ✅ Auto-download results.json ngay sau mỗi domain xong —
#    không đợi tới cuối mới tải, tránh mất hết nếu bị ngắt giữa chừng.

import subprocess
import time
from pathlib import Path
from google.colab import files

EPOCHS = 10
BATCH_SIZE = 64
RESULTS_DIR = Path("experiments/results_pair")

for domain in ['mobile', 'restaurant', 'hotel']:
    config_name = f"pair_{domain}_loss-ce_phobert_mha"
    result_file = RESULTS_DIR / config_name / "results.json"

    print(f"\n{'='*70}")
    print(f"[{domain}] Baseline (C1) — EPOCHS={EPOCHS}")
    print(f"{'='*70}")

    if result_file.exists():
        print(f"⏭️  Đã có kết quả ({result_file}) — bỏ qua, không chạy lại.")
        files.download(str(result_file))
        continue

    cmd = f"python3 train_pair.py --domain {domain} --loss ce --model phobert --epochs {EPOCHS} --batch-size {BATCH_SIZE} --fp16"
    result = subprocess.run(cmd.split(), capture_output=False)

    if result.returncode != 0:
        print(f"❌ ERROR: {domain} failed")
        break

    if result_file.exists():
        print(f"📥 Downloading {result_file} ngay (đề phòng bị ngắt/hết quota sau đây)...")
        files.download(str(result_file))
    else:
        print(f"⚠️  {domain} chạy xong nhưng không thấy results.json ở {result_file} — kiểm tra lại tên config.")

    time.sleep(5)

print("\n✅ Baseline runs complete!")

In [ ]:
# 7. STRATEGY 2: Full ablation — 4 configs × 3 domains
# ✅ Resume-safe: bỏ qua config đã có results.json — chạy lại cell này bao nhiêu
#    lần cũng được sau khi bị ngắt/hết quota, KHÔNG train lại config đã xong.
# ✅ Auto-download results.json ngay sau mỗi (domain, config) — không đợi tới
#    cuối. File results.json rất nhẹ (vài KB) nên tải liên tục không tốn thời gian.

import subprocess
import time
from pathlib import Path
from google.colab import files

EPOCHS = 10
BATCH_SIZE = 64
RESULTS_DIR = Path("experiments/results_pair")

CONFIGS = [
    ("C1_baseline",   "--loss ce"),
    ("C2_norm",       "--loss ce --normalize"),
    ("C3_imbalanced", "--loss focal"),
    ("C4_full",       "--loss focal --normalize"),
]

DOMAINS = ['mobile', 'restaurant', 'hotel']

print(f"Plan: {len(CONFIGS)} configs × {len(DOMAINS)} domains × {EPOCHS} epochs")
print(f"Estimated time: ~{len(CONFIGS)*len(DOMAINS)*EPOCHS//5} minutes (~{len(CONFIGS)*len(DOMAINS)*EPOCHS//5//60} hours)\n")

def expected_result_file(domain, flags):
    norm_suffix = "_norm" if "--normalize" in flags else ""
    loss = "focal" if "focal" in flags else "ce"
    config_name = f"pair_{domain}_loss-{loss}{norm_suffix}_phobert_mha"
    return RESULTS_DIR / config_name / "results.json"

failed = []
for domain in DOMAINS:
    for config_name, flags in CONFIGS:
        result_file = expected_result_file(domain, flags)

        print(f"\n{'='*70}")
        print(f"[{domain}/{config_name}] — {EPOCHS} epochs")
        print(f"{'='*70}")

        if result_file.exists():
            print(f"⏭️  Đã có kết quả ({result_file}) — bỏ qua, không chạy lại.")
            files.download(str(result_file))
            continue

        cmd = f"python3 train_pair.py --domain {domain} {flags} --model phobert --epochs {EPOCHS} --batch-size {BATCH_SIZE} --fp16"
        print(f"Command: {cmd}\n")

        result = subprocess.run(cmd.split(), capture_output=False)
        if result.returncode != 0:
            failed.append(f"{domain}/{config_name}")
            print(f"❌ ERROR: {domain}/{config_name} failed, skipping...")
        elif result_file.exists():
            print(f"📥 Downloading {result_file} ngay (đề phòng bị ngắt/hết quota sau đây)...")
            files.download(str(result_file))
        else:
            print(f"⚠️  Chạy xong nhưng không thấy results.json ở {result_file} — kiểm tra lại tên config.")

        time.sleep(3)

print(f"\n{'='*70}")
print(f"Ablation complete. Failed: {len(failed)}")
if failed:
    print(f"  {failed}")
print(f"{'='*70}")

In [ ]:
# 8. Print summary table
import json
from pathlib import Path
from collections import defaultdict

results_dir = Path("experiments/results_pair")
results = defaultdict(dict)

BASELINE = {"mobile": 61.77, "restaurant": 41.12, "hotel": 52.64}

for results_file in sorted(results_dir.glob("*/results.json")):
    try:
        data = json.load(open(results_file))
        domain = data["domain"]
        config = data["config"]
        test_f1 = data["test"]["macro_f1"] * 100
        results[domain][config] = test_f1
    except Exception as e:
        print(f"[warn] {results_file}: {e}")

# Print table
print("\n" + "="*90)
print("ABLATION RESULTS — macro F1 (3 sentiment classes, loại 'none')")
print("="*90)

header = f"{'Domain':<12} {'C1_baseline':>15} {'C2_norm':>15} {'C3_focal':>15} {'C4_full':>15} {'Baseline':>13}"
print(header)
print("-" * 90)

for domain in ['mobile', 'restaurant', 'hotel']:
    baseline = BASELINE[domain]
    
    # Find results for each config
    c1_val = None
    c2_val = None
    c3_val = None
    c4_val = None
    
    for config_key, f1_val in results.get(domain, {}).items():
        if 'loss-ce_visobert' in config_key and 'norm' not in config_key:
            c1_val = f1_val
        elif 'loss-ce_norm' in config_key:
            c2_val = f1_val
        elif 'loss-focal_visobert' in config_key and 'norm' not in config_key:
            c3_val = f1_val
        elif 'loss-focal_norm' in config_key:
            c4_val = f1_val
    
    c1_str = f"{c1_val:.2f}%" if c1_val else "—"
    c2_str = f"{c2_val:.2f}%" if c2_val else "—"
    c3_str = f"{c3_val:.2f}%" if c3_val else "—"
    c4_str = f"{c4_val:.2f}%" if c4_val else "—"
    
    print(f"{domain:<12} {c1_str:>15} {c2_str:>15} {c3_str:>15} {c4_str:>15} {baseline:>12.2f}%")

print("="*90)
print(f"\n✅ Results saved to: experiments/results_pair/")
print(f"\n📥 Download: Run next cell to download all results as .tar.gz")

In [ ]:
# 9. Download results
from google.colab import files
import os

!tar -czf VITASA_pair_results.tar.gz experiments/results_pair/
!ls -lh VITASA_pair_results.tar.gz

print("\n📥 Downloading results...")
files.download("VITASA_pair_results.tar.gz")
print("✅ Done!")

## Notes

**Nếu timeout:**
1. Colab Free có thể timeout sau 12h hoặc disconnect nếu idle 30 phút
2. Nếu bị interrupt → restart từ cell tiếp theo (data vẫn ở)
3. Hoặc chạy 1 domain/lần thay vì tất cả cùng lúc

**Nếu muốn tăng accuracy:**
- Tăng EPOCHS từ 10 → 20 (nhưng có rủi ro timeout)
- Chạy trên Colab Pro (T4 unlimited hoặc V100)

**Cách lấy results:**
- Download .tar.gz → extract → xem `experiments/results_pair/*/results.json`
- Hoặc print summary từ cell trước khi download
